# Corpus-reliance dose-response — Colab runner

Validates the hidden-hazard corpus-reliance probe by **construction**: teach a charted hazard
into the weights, then sweep LoRA-α and show corpus-reliance falls monotonically as the model
memorizes it. See `DOSE_RESPONSE.md`. Runtime → Change runtime type → **GPU** first.


## 1 · Confirm the GPU


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime → Change runtime type → GPU'


## 2 · Config + clone the repo


In [ ]:
import os

REPO_URL = 'https://github.com/wrgr/socratic-scenarios.git'
BRANCH   = 'claude/gpu-unlearning-experiment-k896wc'   # the branch with the hazard arm

MODEL    = 'Qwen/Qwen2.5-3B-Instruct'   # 3B bf16 (roomy). '7B' auto-enables 4-bit QLoRA.
DTYPE    = 'bfloat16'
ALPHAS   = '0,0.25,0.5,0.75,1.0'        # the LoRA-α knowledge gradient (0 = naive, 1 = taught)
EPOCHS   = '4'
LR       = '1e-4'
BATCH    = '2'                          # small teach set → small batch gives several checkpoints
SAVE_EVERY = '10'                       # checkpoint gradient for the cross-check
LOAD_4BIT = '1' if any(s in MODEL for s in ('7B','8B','13B','14B')) else '0'

%cd /content
# Clone if absent, then hard-sync to the branch tip so a re-run never runs stale code.
![ -d socratic-scenarios ] || git clone --depth 1 --branch $BRANCH $REPO_URL
!cd socratic-scenarios && git fetch --depth 1 origin $BRANCH && git reset --hard FETCH_HEAD
REPO = '/content/socratic-scenarios'; ARM = REPO + '/experiments/unlearning'
%cd $ARM
print(f'config: model={MODEL} dtype={DTYPE} alphas={ALPHAS} load_4bit={LOAD_4BIT}')


## 3 · Install deps (Python + the Node scorer)


In [ ]:
!pip -q install 'transformers>=4.40' 'peft>=0.11' 'accelerate>=0.30' 'safetensors>=0.4' 'bitsandbytes>=0.43'
# Colab ships an old torchao that PEFT's LoRA dispatch rejects; we don't use it — remove it.
!pip -q uninstall -y torchao 2>/dev/null; echo removed-torchao-if-present
# The instrument is TypeScript (npx tsx); install its Node deps once.
!cd $REPO && npm install --no-audit --no-fund --loglevel=error
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 4 · Build the teach set
The location→hazard fact (many phrasings). SFT-ing it in is what moves the model from
corpus-bound (α=0) to leaking (α=1).


In [ ]:
!cd $ARM && python build_hazard_datasets.py && head -1 data/hazard_teach.jsonl


## 5 · Teach the hazard into the weights (SFT)
`--save_every` also snapshots checkpoints for the second (cross-check) gradient.


In [ ]:
import subprocess
cmd = ['python','unlearn.py','--method','sft','--model',MODEL,'--dtype',DTYPE,
       '--sft_file','data/hazard_teach.jsonl','--epochs',EPOCHS,'--lr',LR,
       '--batch_size',BATCH,'--save_every',SAVE_EVERY,'--chat','--out','out/hazard_taught']
if LOAD_4BIT=='1': cmd += ['--load_4bit']
subprocess.run(cmd, cwd=ARM, check=True)


## 6 · Sweep LoRA-α → the corpus-reliance dose-response
One taught adapter, evaluated at each α. **Predicted: corpus-reliance falls monotonically**
as α rises (the model needs the corpus less). If it does not, the instrument isn't measuring
what we claim — a real result, reported, not hidden.


In [ ]:
import subprocess, os
cmd = ['python','dose_response.py','--model',MODEL,'--dtype',DTYPE,
       '--adapter','out/hazard_taught','--alphas',ALPHAS,'--probes','hazard',
       '--out','results/dose_alpha']
if LOAD_4BIT=='1': cmd += ['--load_4bit']
subprocess.run(cmd, cwd=ARM, check=True)   # prints the ASCII curve
print('\n----- results/dose_alpha.csv -----')
print(open(os.path.join(ARM,'results/dose_alpha.csv')).read())


## 7 · (Optional) checkpoint gradient — cross-check the α curve
If the training-checkpoint curve agrees with the α curve, the monotonicity isn't a
gradient-method artifact.


In [ ]:
import glob, os, subprocess
cks = sorted(glob.glob(os.path.join(ARM,'out/hazard_taught/ckpt-*')), key=lambda p:int(p.split('-')[-1]))
print('checkpoints:', [os.path.basename(c) for c in cks])
if len(cks) >= 2:
    cmd = ['python','dose_response.py','--model',MODEL,'--dtype',DTYPE,
           '--checkpoints',','.join(cks),'--probes','hazard','--out','results/dose_ckpt']
    if LOAD_4BIT=='1': cmd += ['--load_4bit']
    subprocess.run(cmd, cwd=ARM, check=True)
else:
    print('too few checkpoints — lower SAVE_EVERY or raise EPOCHS/BATCH and re-teach')


## Done
Send back `results/dose_alpha.csv` (and the checkpoint CSV if you ran it). A monotonic
fall in corpus-reliance is the known-groups validation the single 2×2 cell cannot give.
